In [1]:
import os
import pandas as pd

CAMINHO_RAW = "../data/raw/data.csv"
CAMINHO_OUT = "../data/processed/inspecoes_tratadas.xlsx"

df = pd.read_csv(CAMINHO_RAW)
print("Dimensões (linhas, colunas):", df.shape)

# Contagem de colunas por prefixo (seção)
prefixos = df.columns.to_series().apply(lambda x: x.split("_")[0])
print("\n--- Colunas por seção ---")
print(prefixos.value_counts(sort=False))

notes_cols = [c for c in df.columns if c.endswith("_notes")]
main_cols = [c for c in df.columns if not c.endswith("_notes")]
print(f"\nPrincipais: {len(main_cols)} | _notes: {len(notes_cols)}")

Dimensões (linhas, colunas): (9, 511)

--- Colunas por seção ---
Página de Título                                    3
A - DG - Dados Gerais                              15
B - DD - Dados Turbina                             18
C - EA - Exterior e Acesso                         20
D - S1 - Ground e Virola Interna                   52
E - S2 - Plataforma 2 (T1/T2)                      32
F - S3 - Plataforma 3 (T2/T3)                      32
G - SE - Plataforma do Elevador                    20
H - SY - Torre Seção do Yaw / Yaw deck             40
I - DT - Drivetrain                                28
J - GH - Grupo Hidraúlico                          20
K - GE - Gerador                                   24
L - GB - Gearbox / Multiplicadora                  28
M - ST - Sala do Trafo                             34
N - NI - Nacelle Interior                          48
O - NE - Nacelle Exterior                          22
P - HE - Hub Externo / Pás                         22
Q - HI - Hub Inte

In [2]:
# 1. Classificar as 93 colunas 100% vazias
cols_vazias = df.columns[df.isna().all()].tolist()
def classificar_vazia(c):
    if c.endswith("_notes"):
        return "Coluna _notes vazia"
    if "Comentário Geral" in c or "Outros" in c:
        return "Comentário Geral / Outros"
    return "Item padrão de inspeção/dados"

print(f"Total de colunas 100% vazias: {len(cols_vazias)}")
print(pd.Series([classificar_vazia(c) for c in cols_vazias]).value_counts())

# 2. Filtrar as 235 colunas de inspeção (C até Q, sem _notes)
prefixos_inspecao = tuple(f"{letra} - " for letra in "CDEFGHIJKLMNOPQ")
cols_inspecao = [
    c for c in df.columns
    if c.startswith(prefixos_inspecao) and not c.endswith("_notes")
]
print(f"\nColunas de inspeção (C-Q): {len(cols_inspecao)}")

# 3. Verificar colunas _notes sem par exato (ex: espaço extra)
sem_par = [c for c in notes_cols if c[:-6] not in df.columns]
print("Colunas _notes sem par exato:", sem_par)

Total de colunas 100% vazias: 93
Coluna _notes vazia              61
Comentário Geral / Outros        17
Item padrão de inspeção/dados    15
Name: count, dtype: int64

Colunas de inspeção (C-Q): 235
Colunas _notes sem par exato: ['A - DG - Dados Gerais_Outro pessoal presente (nome/entidade)_notes', 'J - GH - Grupo Hidraúlico _GH.2 - Elétrico/Instrumentação_GH.2.1 - Motor Elétrico _notes']


In [3]:
# Contagem geral de respostas
contagem = pd.Series(df[cols_inspecao].values.ravel()).value_counts(dropna=False)
print("--- Contagem geral (C-Q) ---")
print(contagem)

# Checagem de sanidade fora de cols_inspecao
outras_cols = [c for c in df.columns if c not in cols_inspecao]
grave_fora = df[outras_cols].apply(lambda s: s.astype(str).str.contains("Defeito Grave", na=False)).sum().sum()
imediata_fora = df[outras_cols].apply(lambda s: s.astype(str).str.contains("Ação Imediata!", na=False)).sum().sum()
print(f"\nFora de cols_inspecao -> Defeito Grave: {grave_fora} | Ação Imediata!: {imediata_fora}")

--- Contagem geral (C-Q) ---
Verificado                                                            975
Defeito Ligeiro                                                       539
NaN                                                                   403
Defeito Grave                                                         127
Ação Imediata!                                                         44
Teste ok.                                                              18
Teste realizado por amostragem em dose parafusos cruzados.              5
Realizado por amostragem em dose parafusos de forma cruzada.            2
Teste realizado por amostragem em dose parafusos de forma cruzada.      1
Teste realizado por amostragem em 12 parafusos de forma cruzada.        1
Name: count, dtype: int64

Fora de cols_inspecao -> Defeito Grave: 0 | Ação Imediata!: 0


In [4]:
col_h = "B - DD - Dados Turbina_DD.2 - Estado atuais no momento da inspeção_DD.2.1 - Total Horas de Produção [h] (Out_GridOperationTime)"
col_p = "B - DD - Dados Turbina_DD.2 - Estado atuais no momento da inspeção_DD.2.2 - Total Produção Energia [kWh] (Out_TotalProduction)"

resumo = pd.DataFrame({
    "Turbina": df["Página de Título_ID/Nº Turbina"],
    "Data": df["Página de Título_Realizado em"].str[:10],
    "Horas_h": df[col_h],
    "Energia_MWh": df[col_p],  # Rótulo diz kWh, ordem de grandeza comprova MWh
    "FC_%": ((df[col_p] / (3.465 * df[col_h])) * 100).round(1),
    "Op_Ini": df["A - DG - Dados Gerais_Parada e Energização da Turbina_Início: Turbina operacional/disponível ?"],
    "Op_Fim": df["A - DG - Dados Gerais_Parada e Energização da Turbina_Fim: Turbina operacional/disponível ?"],
    "Warn_Fim": df["A - DG - Dados Gerais_Parada e Energização da Turbina_Fim: Avisos / Warnings presentes ?"],
    "Score": df["score"],
    "TotalScore": df["totalScore"],
    "Ligeiro": (df[cols_inspecao] == "Defeito Ligeiro").sum(axis=1),
    "Grave": (df[cols_inspecao] == "Defeito Grave").sum(axis=1),
    "Imediata": (df[cols_inspecao] == "Ação Imediata!").sum(axis=1),
}).sort_values("Turbina")

print(resumo.to_string(index=False))

 Turbina       Data  Horas_h  Energia_MWh  FC_% Op_Ini Op_Fim Warn_Fim  Score  TotalScore  Ligeiro  Grave  Imediata
       1 2024-08-08    17213        24241  40.6    Sim    Sim      Não    105         205       63     15         6
       2 2024-07-29    19530        31657  46.8    Sim    Sim      Não    118         205       52     13         6
       3 2024-07-31    19337        33372  49.8    Sim    Não      Sim    115         203       54     11         8
       4 2024-08-07    18694        36738  56.7    Sim    Sim      Não    103         207       68     15         5
       5 2024-08-01    18998        35551  54.0    Sim    Sim      Não    108         205       57     13        11
       6 2024-08-02    18833        31279  47.9    Sim    Sim      Não    114         204       54     16         4
       7 2024-08-05    17551        25475  41.9    Sim    Sim      Não    113         209       60     18         1
       8 2024-08-06    18776        27537  42.3    Sim    Sim      Não  

In [5]:
def pegar_nota(row, col):
    for sufixo in ["_notes", " _notes"]:
        if col + sufixo in row.index and pd.notna(row[col + sufixo]):
            return str(row[col + sufixo]).replace("\n", " | ")
    return ""

registros = []
for _, row in df.iterrows():
    t = row["Página de Título_ID/Nº Turbina"]
    for col in cols_inspecao:
        st = row[col]
        if st in ["Ação Imediata!", "Defeito Grave", "Defeito Ligeiro"]:
            registros.append({
                "Turbina": t,
                "Secao": col.split("_")[0],
                "Item": col.split("_")[-1],
                "Status": st,
                "Nota": pegar_nota(row, col),
            })

df_def = pd.DataFrame(registros)

tabela_secao = pd.crosstab(df_def["Secao"], df_def["Status"])[["Ação Imediata!", "Defeito Grave", "Defeito Ligeiro"]]
tabela_secao["Total"] = tabela_secao.sum(axis=1)
print(tabela_secao.sort_values(["Ação Imediata!", "Defeito Grave"], ascending=False).to_string())

Status                                           Ação Imediata!  Defeito Grave  Defeito Ligeiro  Total
Secao                                                                                                 
N - NI - Nacelle Interior                                     8             21               59     88
L - GB - Gearbox / Multiplicadora                             8              7               40     55
D - S1 - Ground e Virola Interna                              7              2               41     50
M - ST - Sala do Trafo                                        6             11               20     37
Q - HI - Hub Interno / Sistema Hidráulico / Pás               5             41               74    120
H - SY - Torre Seção do Yaw / Yaw deck                        4             22               55     81
I - DT - Drivetrain                                           3              2               64     69
G - SE - Plataforma do Elevador                               2          

In [6]:
imediatas = df_def[df_def["Status"] == "Ação Imediata!"].sort_values(["Secao", "Turbina"])
for i, (_, r) in enumerate(imediatas.iterrows(), 1):
    print(f"{i:02d}. [T{r['Turbina']}] {r['Secao']} | {r['Item']}")
    print(f"    -> {r['Nota']}")

01. [T1] D - S1 - Ground e Virola Interna | S1.5.2 - Linha de Vida (Certificação, Validação, estado geral)
    -> Data de validade da inspeção expirada.  | Realizamos uma inspeção em local e vimos que podemos subir.
02. [T1] D - S1 - Ground e Virola Interna | S1.5.4 - Escada de acesso principal (Certificação, Validação, estado geral)
    -> Data de validade da inspeção expirada.  | Realizamos uma inspeção em local e vimos que podemos subir.
03. [T2] D - S1 - Ground e Virola Interna | S1.5.8 - Extintor de combate a princípio de incêndio (Carga, 
capacidade, validade, estado geral)
    -> Sem extintor.
04. [T5] D - S1 - Ground e Virola Interna | S1.4.1 - Elevador de serviço -  (Validação e certificação para uso)
    -> Data de validade da inspeção expirada.
05. [T5] D - S1 - Ground e Virola Interna | S1.5.2 - Linha de Vida (Certificação, Validação, estado geral)
    -> Data de validade da inspeção expirada.
06. [T5] D - S1 - Ground e Virola Interna | S1.5.4 - Escada de acesso principal (

In [7]:
# Top 15 Defeito Grave
df_gr = df_def[df_def["Status"] == "Defeito Grave"]
ranking_gr = df_gr.groupby(["Secao", "Item"]).size().sort_values(ascending=False)
print("--- Top 15 itens com Defeito Grave ---")
print(ranking_gr.head(15).to_string())

print("\n--- Detalhes dos Top 5 itens (Defeito Grave) ---")
for sec, it in ranking_gr.head(5).index:
    print(f"\n>> {sec} | {it}")
    sub = df_gr[(df_gr["Secao"] == sec) & (df_gr["Item"] == it)].sort_values("Turbina")
    for _, r in sub.iterrows():
        print(f"   [T{r['Turbina']}] {r['Nota']}")

# Top 5 Defeito Ligeiro
df_lig = df_def[df_def["Status"] == "Defeito Ligeiro"]
print("\n--- Top 5 itens com Defeito Ligeiro ---")
print(df_lig.groupby(["Secao", "Item"]).size().sort_values(ascending=False).head(5).to_string())

--- Top 15 itens com Defeito Grave ---
Secao                                            Item                                                                                                                                                                                             
H - SY - Torre Seção do Yaw / Yaw deck           Não conformidades                                                                                                                                                                                    8
Q - HI - Hub Interno / Sistema Hidráulico / Pás  HI.2.3 - Gabinetes (cabos, limpeza, estado geral)                                                                                                                                                    8
O - NE - Nacelle Exterior                        NE.2.1 - Luzes de balizamento (Diurna - branco / Noturna - vermelho, suportes, cabos, conexões,  posicionamento, danos por raios, integridade, estado geral)                 

In [8]:
os.makedirs("../data/processed", exist_ok=True)
df_raw = pd.read_csv(CAMINHO_RAW, keep_default_na=False)

def pegar_nota_raw(row, col):
    for sufixo in ["_notes", " _notes"]:
        if col + sufixo in row.index and str(row[col + sufixo]).strip() != "":
            return str(row[col + sufixo]).strip()
    return ""

fato_rows = []
for _, row in df_raw.iterrows():
    t = int(row["Página de Título_ID/Nº Turbina"])
    for col in cols_inspecao:
        st = str(row[col]).strip()
        nota = pegar_nota_raw(row, col)
        if st != "" or nota != "":
            partes = col.split("_")
            fato_rows.append({
                "Turbina": t,
                "Secao": partes[0],
                "Subgrupo": partes[1] if len(partes) > 2 else "",
                "Item": partes[-1],
                "Status": st if st != "" else "Sem Status",
                "Eh_Nao_Conformidade": "Sim" if st in ["Ação Imediata!", "Defeito Grave", "Defeito Ligeiro"] else "Não",
                "Nota": nota,
            })

df_fato = pd.DataFrame(fato_rows).sort_values(["Turbina", "Secao"])

dim_rows = []
for _, row in df_raw.iterrows():
    t = int(row["Página de Título_ID/Nº Turbina"])
    horas = float(row[col_h])
    prod_mwh = float(row[col_p])
    sub = df_fato[df_fato["Turbina"] == t]
    dim_rows.append({
        "Turbina": t,
        "Data_Inspecao": str(row["Página de Título_Realizado em"])[:10],
        "Modelo": row["B - DD - Dados Turbina_DD.1 - Dados gerais_DD.1.1 - Modelo da turbina"],
        "Potencia_Nominal_kW": 3465,
        "Horas_Producao_h": int(horas),
        "Energia_Produzida_MWh": int(prod_mwh),
        "Fator_Capacidade_Estimado_%": round((prod_mwh / (3.465 * horas)) * 100, 1),
        "Op_Inicio": row["A - DG - Dados Gerais_Parada e Energização da Turbina_Início: Turbina operacional/disponível ?"],
        "Warn_Inicio": row["A - DG - Dados Gerais_Parada e Energização da Turbina_Início: Avisos / Warnings presentes ?"],
        "Op_Fim": row["A - DG - Dados Gerais_Parada e Energização da Turbina_Fim: Turbina operacional/disponível ?"],
        "Warn_Fim": row["A - DG - Dados Gerais_Parada e Energização da Turbina_Fim: Avisos / Warnings presentes ?"],
        "Score": int(row["score"]),
        "Total_Score": int(row["totalScore"]),
        "Acao_Imediata": int((sub["Status"] == "Ação Imediata!").sum()),
        "Defeito_Grave": int((sub["Status"] == "Defeito Grave").sum()),
        "Defeito_Ligeiro": int((sub["Status"] == "Defeito Ligeiro").sum()),
        "Verificado": int((sub["Status"] == "Verificado").sum()),
        "NA": int((sub["Status"] == "N/A").sum()),
    })

df_dim = pd.DataFrame(dim_rows).sort_values("Turbina")

df_nc = df_fato[df_fato["Eh_Nao_Conformidade"] == "Sim"]
df_secoes = pd.crosstab(df_nc["Secao"], df_nc["Status"])[["Ação Imediata!", "Defeito Grave", "Defeito Ligeiro"]]
df_secoes["Total_NC"] = df_secoes.sum(axis=1)
df_secoes = df_secoes.sort_values(["Ação Imediata!", "Defeito Grave"], ascending=False).reset_index()

with pd.ExcelWriter(CAMINHO_OUT, engine="openpyxl") as writer:
    df_dim.to_excel(writer, sheet_name="Dim_Turbinas", index=False)
    df_fato.to_excel(writer, sheet_name="Fato_Inspecoes", index=False)
    df_secoes.to_excel(writer, sheet_name="Resumo_Secoes", index=False)

print(f"Salvo com sucesso em: {CAMINHO_OUT}")

Salvo com sucesso em: ../data/processed/inspecoes_tratadas.xlsx
